# Success rate comparison: Chinese vs German

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import requests
from datetime import datetime, timedelta


In [ ]:
# with opened ANKI
ANKI_URL = "http://127.0.0.1:8765"
DECKS = {
    "chinese": "#2a78d6",  # blue
    "german": "#1baf7a",   # aqua
}
DAYS_BACK = 365          # 30 for last month, 365 for last year


In [ ]:
def load_reviews(deck_name, days_back):

    cutoff_timestamp_ms = int((datetime.now() - timedelta(days=days_back)).timestamp() * 1000)

    payload = {
        "action": "cardReviews",
        "version": 6,
        "params": {
            "deck": deck_name,
            "startID": cutoff_timestamp_ms
        }
    }

    try:
        response = requests.post(ANKI_URL, json=payload).json()
    except requests.exceptions.ConnectionError:
        print("Error: Anki is not running or AnkiConnect is not installed.")
        exit(1)

    if response.get("error"):
        print(f"API Error: {response['error']}")
        exit(1)

    reviews = response["result"]

    # Group by day
    daily_stats = {}
    for entry in reviews:
        # entry structure: [review_time_ms, card_id, usn, ease, interval, last_interval, factor, time, type]
        rev_id, card_id, _, ease = entry[:4]

        day_str = datetime.fromtimestamp(rev_id / 1000).strftime('%Y-%m-%d')
        if day_str not in daily_stats:
            daily_stats[day_str] = {"pass": 0, "fail": 0}

        if ease == 1:
            daily_stats[day_str]["fail"] += 1
        else:
            daily_stats[day_str]["pass"] += 1

    # Create a DataFrame from the daily stats
    df = pd.DataFrame.from_dict(daily_stats, orient='index')
    df.index = pd.to_datetime(df.index)
    df = df.sort_index()

    df["pct"] = df["pass"] / (df["pass"] + df["fail"]) * 100
    df["date"] = df.index
    df.reset_index(drop=True, inplace=True)

    # Days relative to today (negative = in the past), so decks with different
    # review-history lengths still line up on a shared, real time axis.
    today = pd.Timestamp(datetime.now().date())
    df["n_day"] = (df["date"] - today).dt.days

    df["pct_rolling_7"] = df["pct"].rolling(window=7).mean()

    # drop outliers (more than 3 std from the mean) so a single noisy day
    # doesn't stretch the shared y-axis
    mean_pct, std_pct = df["pct"].mean(), df["pct"].std()
    df = df[(df["pct"] <= mean_pct + 3 * std_pct) & (df["pct"] >= mean_pct - 3 * std_pct)]

    return df


In [ ]:
deck_data = {deck: load_reviews(deck, DAYS_BACK) for deck in DECKS}
{deck: df.tail() for deck, df in deck_data.items()}


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

for deck, color in DECKS.items():
    df = deck_data[deck]
    ax.scatter(df["n_day"], df["pct"], color=color, alpha=0.15, s=20)
    ax.plot(df["n_day"], df["pct_rolling_7"], color=color, linewidth=2, label=f"{deck} (7-day avg)")

ax.set_title("Daily % Correct \u2014 Chinese vs German (7-day rolling average)")
ax.set_xlabel("Days ago")
ax.set_ylabel("% Correct")
ax.set_ylim(0, 100)
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()
